# 🛡️ Face Anti-Spoofing dengan CDCN
## Central Difference Convolutional Network — Pipeline Lengkap

**Paper:** Yu, Z. et al. *Searching Central Difference Convolutional Networks for Face Anti-Spoofing.* CVPR 2020. [[arXiv]](https://arxiv.org/pdf/2003.04092)

**Dataset:** [Real vs Fake Anti-Spoofing — Kaggle](https://www.kaggle.com/datasets/trainingdatapro/real-vs-fake-anti-spoofing-video-classification)

---

## 🗺️ Alur Pipeline

```
Raw Video → Ekstrak Frame → EDA → Preprocessing →
Arsitektur CDCN → Training → Evaluasi Test Set →
Overfitting Analysis → Ablation Study → Inference
```

## ⚙️ Cara Menjalankan
1. Aktifkan GPU: **Runtime → Change runtime type → T4 GPU**
2. Jalankan semua cell berurutan: **Runtime → Run all**


---
## ⚙️ Bagian 0: Setup Environment

### Apa yang dilakukan?
Menginstall library tambahan dan memverifikasi bahwa GPU tersedia.

### Mengapa perlu GPU?
Model CDCN memiliki ~5 juta parameter. Training di CPU bisa berjam-jam,
sedangkan dengan GPU T4 selesai dalam ~15 menit.


In [ ]:
import subprocess, sys

def pip_install(pkg):
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q",
                        "--disable-pip-version-check", "--no-deps"],
                       capture_output=True, text=True)
    return r.returncode == 0

for name, pkg in {"facenet-pytorch": "facenet-pytorch", "kaggle": "kaggle"}.items():
    print(f"  {'✅' if pip_install(pkg) else '⚠️ '} {name}")
print("\n✅ Install selesai!")

  ✅ facenet-pytorch
  ✅ kaggle

✅ Install selesai!


In [ ]:
import sys, torch, numpy as np, cv2
import pandas as pd, matplotlib, seaborn, sklearn

print("=" * 50)
print("🔧 ENVIRONMENT")
print("=" * 50)
print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch   : {torch.__version__}")
print(f"Numpy     : {np.__version__}")
print(f"OpenCV    : {cv2.__version__}")
print(f"CUDA      : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    DEVICE = torch.device("cuda")
    print("\n✅ GPU tersedia!")
else:
    print("\n⚠️  Aktifkan GPU: Runtime → Change runtime type → T4 GPU")
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")
print("=" * 50)

🔧 ENVIRONMENT
Python    : 3.12.13
PyTorch   : 2.11.0+cu128
Numpy     : 2.0.2
OpenCV    : 4.13.0
CUDA      : True
GPU       : Tesla T4
VRAM      : 15.6 GB

✅ GPU tersedia!
Device: cuda


---
## 📥 Bagian 1: Download Dataset

### Tentang Dataset
Dataset **Real vs Fake Anti-Spoofing** dari Kaggle berisi:
- **Real**: rekaman wajah manusia asli yang direkam langsung
- **Attack (Fake)**: video wajah yang diputar di layar ponsel lalu direkam ulang (replay attack)

### Cara mendapat kaggle.json
1. Login ke kaggle.com → Settings → API → **Create New Token**
2. File `kaggle.json` terunduh otomatis → upload saat diminta


In [ ]:
import os
from google.colab import files

print("📤 Upload kaggle.json:")
uploaded = files.upload()

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.system("mv kaggle.json ~/.kaggle/ 2>/dev/null; chmod 600 ~/.kaggle/kaggle.json")
print("✅ Kaggle API siap!")

DATA_ROOT = "/content/dataset"
os.makedirs(DATA_ROOT, exist_ok=True)

ret = os.system(
    "kaggle datasets download "
    "-d trainingdatapro/real-vs-fake-anti-spoofing-video-classification "
    f"-p {DATA_ROOT} --unzip -q"
)
print("✅ Download selesai!" if ret == 0 else "⚠️  Gagal download")

print("\n📂 Struktur folder:")
for root, dirs, files_ in os.walk(DATA_ROOT):
    depth = root.replace(DATA_ROOT, "").count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(root) + f"/  ({len(files_)} files)")

📤 Upload kaggle.json:


KeyboardInterrupt: 

---
## 🔍 Bagian 2: Eksplorasi Data (EDA)

### Apa itu EDA?
EDA adalah proses memahami data sebelum membangun model. Kita perlu tahu:
- Berapa banyak data yang tersedia?
- Apakah kelas seimbang (jumlah real ≈ fake)?
- Seperti apa tampilan visual datanya?

### ⚠️ Kenapa Split di Level VIDEO, bukan Frame?

Ini adalah hal yang sangat penting dan sering diabaikan.

Satu video menghasilkan banyak frame. Kalau split dilakukan di level frame,
frame dari orang yang SAMA bisa masuk ke training dan testing sekaligus.
Model akan "mengenali wajah" bukan "belajar liveness detection".
Ini disebut **data leakage** dan menyebabkan hasil evaluasi yang tidak jujur.

```
❌ SALAH (frame-level split):
   Video A → frame 1 ke train, frame 2 ke test (orang sama!)

✅ BENAR (video-level split):
   Video A → SEMUA framenya ke train ATAU SEMUA ke test
   Video B → orang berbeda, semua ke test
```

Dengan video-level split, test set berisi orang yang BELUM PERNAH
dilihat model — ini adalah evaluasi yang jujur.


In [ ]:
import os, cv2, glob, warnings, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

DATA_ROOT  = "/content/dataset"
FRAMES_DIR = "/content/frames"
os.makedirs(f"{FRAMES_DIR}/real", exist_ok=True)
os.makedirs(f"{FRAMES_DIR}/fake", exist_ok=True)

REAL_KW = ["real","live","genuine","bona"]
FAKE_KW = ["fake","spoof","attack","print","replay"]

def get_label_from_path(path):
    """Label dari nama folder terdekat — lebih aman dari cek full path."""
    for part in reversed(Path(path).parts[:-1]):
        p = part.lower()
        if any(k in p for k in FAKE_KW): return "fake"
        if any(k in p for k in REAL_KW): return "real"
    return None

real_images, fake_images, video_files = [], [], []
for root, _, files in os.walk(DATA_ROOT):
    for f in files:
        path = os.path.join(root, f)
        ext  = f.lower()
        if ext.endswith((".jpg",".jpeg",".png")):
            lbl = get_label_from_path(path)
            if   lbl == "real": real_images.append(path)
            elif lbl == "fake": fake_images.append(path)
        elif ext.endswith((".mp4",".avi",".mov",".mkv")):
            video_files.append(path)

def extract_frames(video_path, out_dir, label, max_frames=25, interval=8):
    """
    Ekstrak frame dari video dengan interval tertentu.
    Interval=8: menghindari frame yang terlalu mirip (hampir identik).
    Max=25: menjaga dataset seimbang antar video yang durasinya berbeda.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return 0
    stem = Path(video_path).stem
    count = saved = 0
    while cap.isOpened() and saved < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if count % interval == 0:
            cv2.imwrite(f"{out_dir}/{label}/{stem}_f{count:05d}.jpg", frame)
            saved += 1
        count += 1
    cap.release()
    return saved

if video_files:
    print(f"🎬 Mengekstrak frame dari {len(video_files)} video...")
    total = 0
    for vp in tqdm(video_files, desc="Ekstrak"):
        lbl = get_label_from_path(vp)
        if lbl:
            total += extract_frames(vp, FRAMES_DIR, lbl)
    real_images += glob.glob(f"{FRAMES_DIR}/real/*.jpg")
    fake_images += glob.glob(f"{FRAMES_DIR}/fake/*.jpg")
    print(f"✅ {total:,} frame diekstrak")

print(f"\n{'='*45}")
print(f"✅ Real  : {len(real_images):,} frame")
print(f"❌ Fake  : {len(fake_images):,} frame")
print(f"📊 Total : {len(real_images)+len(fake_images):,} frame")
print(f"{'='*45}")

In [ ]:
# ── Split di level VIDEO (mencegah data leakage) ─────────────

def get_video_stem(frame_path):
    """
    Ekstrak nama video asal dari nama frame.
    Contoh: person01_real_f00008.jpg → person01_real
    """
    name  = Path(frame_path).stem
    parts = name.split("_")
    for i in range(len(parts)-1, -1, -1):
        if parts[i].startswith("f") and parts[i][1:].isdigit():
            return "_".join(parts[:i])
    return name

# Mapping: video_stem → list of (frame_path, label)
video_to_frames = defaultdict(list)
for path, label in zip(real_images + fake_images,
                        [1]*len(real_images) + [0]*len(fake_images)):
    stem = get_video_stem(path)
    video_to_frames[stem].append((path, label))

video_stems  = list(video_to_frames.keys())
video_labels = [video_to_frames[s][0][1] for s in video_stems]

print(f"📹 Total video unik : {len(video_stems)}")
print(f"   Real  : {sum(video_labels)}")
print(f"   Fake  : {len(video_labels)-sum(video_labels)}")

# Split video (bukan frame!)
vid_tv, vid_test, _, _ = train_test_split(
    video_stems, video_labels,
    test_size=0.15, stratify=video_labels, random_state=42)
vid_train, vid_val, _, _ = train_test_split(
    vid_tv,
    [video_to_frames[s][0][1] for s in vid_tv],
    test_size=0.176,
    stratify=[video_to_frames[s][0][1] for s in vid_tv],
    random_state=42)

# Expand video → frame
def expand(vlist):
    paths, labels = [], []
    for stem in vlist:
        for path, label in video_to_frames[stem]:
            paths.append(path); labels.append(label)
    return paths, labels

X_train, y_train = expand(vid_train)
X_val,   y_val   = expand(vid_val)
X_test,  y_test  = expand(vid_test)

print(f"\n📊 Split VIDEO-LEVEL (anti data leakage):")
print(f"   Train : {len(X_train):,} frame dari {len(vid_train)} video")
print(f"   Val   : {len(X_val):,} frame dari {len(vid_val)} video")
print(f"   Test  : {len(X_test):,} frame dari {len(vid_test)} video")
print(f"\n✅ Orang di test set = orang baru yang belum dilihat model")

In [ ]:
# ── Visualisasi EDA ───────────────────────────────────────────
import random

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
n_real = len(real_images); n_fake = len(fake_images)
counts = [n_real, n_fake]; colors = ["#2ECC71","#E74C3C"]

ax = axes[0,0]
bars = ax.bar(["Real","Fake"], counts, color=colors, edgecolor="white", lw=1.5)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(counts)*0.01,
            f"{cnt:,}", ha="center", fontweight="bold")
ax.set_title("Distribusi Kelas", fontweight="bold")
ax.set_ylabel("Jumlah Frame")

ax = axes[0,1]
ax.pie(counts, labels=["Real","Fake"], autopct="%1.1f%%",
       colors=colors, startangle=90,
       wedgeprops=dict(edgecolor="white", lw=2))
ax.set_title(f"Proporsi Kelas\nImbalance ratio: {max(counts)/min(counts):.2f}x",
             fontweight="bold")

ax = axes[0,2]
sc = [len(X_train), len(X_val), len(X_test)]
sc_colors = ["#3498DB","#F39C12","#9B59B6"]
bars3 = ax.bar(["Train","Val","Test"], sc, color=sc_colors, alpha=0.85)
for bar, cnt in zip(bars3, sc):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            f"{cnt:,}", ha="center", fontweight="bold")
ax.set_title("Distribusi Split\n(Video-Level Split)", fontweight="bold")

for idx, (paths, lbl, col) in enumerate([
    (real_images, "🟢 REAL  (Wajah Asli)", "#2ECC71"),
    (fake_images, "🔴 FAKE  (Layar HP)", "#E74C3C")]):
    ax = axes[1, idx]
    img = cv2.imread(random.choice(paths))
    if img is not None:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(lbl, fontweight="bold", color=col, fontsize=12)
    ax.axis("off")
    for sp in ax.spines.values():
        sp.set_edgecolor(col); sp.set_linewidth(4)

ax = axes[1,2]; ax.axis("off")
info = (f"INFORMASI DATASET\n{'─'*26}\n\n"
        f"Total video   : {len(video_stems)}\n"
        f"Total frame   : {n_real+n_fake:,}\n\n"
        f"Split Strategy\n: Video-Level\n(anti data leakage)\n\n"
        f"Train videos  : {len(vid_train)}\n"
        f"Val   videos  : {len(vid_val)}\n"
        f"Test  videos  : {len(vid_test)}\n\n"
        f"Imbalance     : {max(counts)/min(counts):.2f}x")
ax.text(0.1, 0.95, info, transform=ax.transAxes, fontsize=10,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="#F8F9FA", edgecolor="#CCCCCC"))

plt.suptitle("Exploratory Data Analysis (EDA)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/eda.png", bbox_inches="tight", dpi=120)
plt.show()
print("✅ Disimpan: /content/eda.png")

### 📌 Interpretasi Hasil EDA

**Distribusi Kelas**
- Idealnya imbalance ratio mendekati 1.0 (seimbang)
- Kalau tidak seimbang, model cenderung bias ke kelas yang lebih banyak

**Contoh Visual**
- Gambar **REAL**: wajah langsung, tidak ada batas layar, warna natural
- Gambar **FAKE**: ada batas layar ponsel, pantulan cahaya, warna sedikit berbeda karena kamera merekam layar

**Split Video-Level**
- Jumlah video di train/val/test mewakili orang yang benar-benar berbeda
- Ini memastikan evaluasi di test set adalah evaluasi yang jujur


---
## 🔄 Bagian 3: Preprocessing & DataLoader

### Tahapan Preprocessing

**1. Resize ke 224×224 piksel**
Model CNN membutuhkan input ukuran konsisten.
224×224 adalah standar umum: cukup besar untuk detail wajah, tidak terlalu besar.

**2. Normalisasi ImageNet**
Nilai piksel 0-255 dinormalisasi menggunakan mean dan std ImageNet:
- Mean = [0.485, 0.456, 0.406]
- Std  = [0.229, 0.224, 0.225]

Mengapa ImageNet? Nilai ini terbukti membuat training CNN lebih stabil.

**3. Augmentasi (hanya untuk Training)**
Memperbanyak variasi data agar model tidak hafal gambar training.

> **Aturan penting:** Augmentasi HANYA untuk training.
> Val dan test harus mencerminkan kondisi data nyata tanpa modifikasi.


In [ ]:
import torch, cv2, numpy as np, random
from torch.utils.data import Dataset, DataLoader

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def augment_train(img_bgr):
    """Preprocessing + Augmentasi untuk data TRAINING saja."""
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    if random.random() > 0.5:
        img = cv2.flip(img, 1)
    if random.random() > 0.5:
        alpha = 1 + random.uniform(-0.3, 0.3)
        beta  = random.uniform(-30, 30)
        img   = np.clip(img.astype(np.float32)*alpha + beta, 0, 255).astype(np.uint8)
    if random.random() > 0.8:
        ks  = random.choice([3, 5])
        img = cv2.GaussianBlur(img, (ks, ks), 0)
    if random.random() > 0.8:
        noise = np.random.normal(0, random.uniform(3,15), img.shape).astype(np.float32)
        img   = np.clip(img.astype(np.float32)+noise, 0, 255).astype(np.uint8)
    img = img.astype(np.float32) / 255.0
    img = (img - MEAN) / STD
    return torch.from_numpy(img.transpose(2,0,1)).float()

def augment_val(img_bgr):
    """Preprocessing TANPA augmentasi untuk Val dan Test."""
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = img.astype(np.float32) / 255.0
    img = (img - MEAN) / STD
    return torch.from_numpy(img.transpose(2,0,1)).float()


class AntispoofDataset(Dataset):
    """
    Dataset mengembalikan 3 item per sampel:
    1. Tensor gambar (3 x 224 x 224)
    2. Label klasifikasi (1.0=real, 0.0=fake)
    3. Depth map target (1 x 28 x 28)

    Depth map: real=1.0 (ada struktur 3D), fake=0.0 (layar datar 2D).
    Ini digunakan sebagai auxiliary supervision untuk membantu model
    belajar representasi kedalaman, bukan hanya klasifikasi warna.
    """
    def __init__(self, paths, labels, train=True, depth_size=28):
        self.paths = paths; self.labels = labels
        self.train = train; self.dsize  = depth_size

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        if img is None:
            img = np.zeros((224,224,3), dtype=np.uint8)
        tensor    = augment_train(img) if self.train else augment_val(img)
        lbl       = float(self.labels[idx])
        depth_val = 1.0 if lbl == 1.0 else 0.0
        depth     = torch.full((1, self.dsize, self.dsize), depth_val)
        return tensor, torch.tensor(lbl, dtype=torch.float32), depth


BATCH = 32
train_loader = DataLoader(AntispoofDataset(X_train,y_train,train=True),
                          BATCH, shuffle=True,  num_workers=2,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(AntispoofDataset(X_val,  y_val,  train=False),
                          BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(AntispoofDataset(X_test, y_test, train=False),
                          BATCH, shuffle=False, num_workers=2, pin_memory=True)

n_r = sum(y_train); n_f = len(y_train)-n_r
print(f"Train : {len(X_train):,} frame | Real:{n_r} Fake:{n_f}")
print(f"Val   : {len(X_val):,} frame")
print(f"Test  : {len(X_test):,} frame")
print("\n✅ DataLoader siap!")

In [ ]:
# Visualisasi augmentasi — tunjukkan gambar yang sama bisa tampak berbeda
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
sample_real = cv2.imread(real_images[0])
sample_fake = cv2.imread(fake_images[0])

axes[0,0].imshow(cv2.cvtColor(cv2.resize(sample_real,(224,224)),cv2.COLOR_BGR2RGB))
axes[0,0].set_title("Real — Original", fontweight="bold"); axes[0,0].axis("off")

axes[1,0].imshow(cv2.cvtColor(cv2.resize(sample_fake,(224,224)),cv2.COLOR_BGR2RGB))
axes[1,0].set_title("Fake — Original", fontweight="bold", color="#E74C3C")
axes[1,0].axis("off")

for i in range(1, 5):
    for row, (sample, color) in enumerate([(sample_real,"#27AE60"),(sample_fake,"#E74C3C")]):
        t = augment_train(sample).numpy().transpose(1,2,0)
        t = (t * STD + MEAN).clip(0, 1)
        axes[row,i].imshow(t)
        axes[row,i].set_title(f"Augmented #{i}", fontweight="bold", color=color)
        axes[row,i].axis("off")

plt.suptitle("Visualisasi Augmentasi — Gambar yang sama tampak berbeda setiap epoch\n"
             "(Augmentasi membuat model lebih robust, mengurangi overfitting)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/augmentasi.png", bbox_inches="tight", dpi=120)
plt.show()
print("✅ Disimpan: /content/augmentasi.png")

---
## 🏗️ Bagian 4: Arsitektur Model CDCN

### Mengapa CDCN, bukan CNN biasa?

CNN biasa mengekstrak fitur dari nilai intensitas piksel. Masalahnya,
nilai intensitas wajah asli dan layar HP bisa sangat mirip — keduanya
menampilkan wajah manusia.

**CDCN** menambahkan kemampuan mengekstrak **gradien lokal** (perubahan
intensitas antar piksel tetangga). Gradien ini mencerminkan **tekstur
permukaan** — dan tekstur kulit wajah nyata berbeda dari tekstur layar.

### Analogi Sederhana
Bayangkan Anda memegang foto wajah dan wajah asli di bawah lampu.
Dengan sentuhan jari, Anda merasakan perbedaan:
- Kertas foto: halus dan rata (2D)
- Kulit wajah: ada tekstur pori, kontour hidung, pipi (3D)

CDC melakukan hal ini secara digital — mendeteksi gradien yang
mencerminkan ada/tidaknya struktur 3D.

### Formula CDC
```
Output = θ × (Gradient Info) + (1−θ) × (Intensity Info)
```
- θ = 0.0 → konvolusi biasa
- θ = 0.7 → nilai optimal dari paper (70% gradient + 30% intensity)
- θ = 1.0 → pure gradient


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class Conv2d_cd(nn.Module):
    """Central Difference Convolution — inti dari CDCN."""
    def __init__(self, in_ch, out_ch, ks=3, stride=1, pad=1,
                 dilation=1, groups=1, bias=False, theta=0.7):
        super().__init__()
        self.conv  = nn.Conv2d(in_ch, out_ch, ks, stride, pad, dilation, groups, bias)
        self.theta = theta

    def forward(self, x):
        out = self.conv(x)
        if abs(self.theta) < 1e-8: return out
        kd  = self.conv.weight.sum(2).sum(2)[:,:,None,None]
        return out - self.theta * F.conv2d(x, kd, self.conv.bias,
                                           self.conv.stride, 0,
                                           groups=self.conv.groups)

class CDCBlock(nn.Module):
    """
    Residual Block dengan CDC Layer.
    Residual connection mencegah vanishing gradient dan memudahkan
    training model yang dalam.
    """
    def __init__(self, in_ch, out_ch, theta=0.7):
        super().__init__()
        self.c1   = Conv2d_cd(in_ch, out_ch, theta=theta)
        self.b1   = nn.BatchNorm2d(out_ch)
        self.c2   = Conv2d_cd(out_ch, out_ch, theta=theta)
        self.b2   = nn.BatchNorm2d(out_ch)
        self.skip = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False),
                                   nn.BatchNorm2d(out_ch))
                     if in_ch != out_ch else nn.Identity())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        r = self.skip(x)
        x = self.relu(self.b1(self.c1(x)))
        return self.relu(self.b2(self.c2(x)) + r)

class CDCN(nn.Module):
    """
    Central Difference Convolutional Network (Yu et al., CVPR 2020).

    Dua output:
    - cls_head : prediksi probabilitas real/fake (main task)
    - depth_head: estimasi depth map 28x28 (auxiliary task)

    Auxiliary depth supervision memaksa model belajar representasi
    kedalaman 3D, bukan sekadar mengklasifikasi warna.
    """
    def __init__(self, theta=0.7):
        super().__init__()
        self.stem = nn.Sequential(
            Conv2d_cd(3,64,theta=theta),nn.BatchNorm2d(64),nn.ReLU(True),
            Conv2d_cd(64,64,theta=theta),nn.BatchNorm2d(64),nn.ReLU(True))
        self.s1 = nn.Sequential(nn.MaxPool2d(2), CDCBlock(64,128,theta))
        self.s2 = nn.Sequential(nn.MaxPool2d(2), CDCBlock(128,256,theta))
        self.s3 = nn.Sequential(nn.MaxPool2d(2), CDCBlock(256,512,theta))
        self.depth_head = nn.Sequential(
            nn.Conv2d(512,128,1),nn.BatchNorm2d(128),nn.ReLU(True),
            nn.Conv2d(128,1,1),nn.Sigmoid())
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),nn.Flatten(),
            nn.Linear(512,256),nn.BatchNorm1d(256),nn.ReLU(True),
            nn.Dropout(0.5),nn.Linear(256,1))
        self._init_w()

    def _init_w(self):
        for m in self.modules():
            if isinstance(m, Conv2d_cd):
                nn.init.kaiming_normal_(m.conv.weight, nonlinearity="relu")
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d,nn.BatchNorm1d)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.zeros_(m.bias)

    def forward(self, x):
        f = self.s3(self.s2(self.s1(self.stem(x))))
        return self.cls_head(f), self.depth_head(f)

class BaselineCNN(nn.Module):
    """Vanilla CNN tanpa CDC — pembanding untuk mengukur kontribusi CDC."""
    def __init__(self):
        super().__init__()
        def blk(ic,oc):
            return nn.Sequential(
                nn.Conv2d(ic,oc,3,1,1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True),
                nn.Conv2d(oc,oc,3,1,1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True))
        self.stem=blk(3,64); self.s1=nn.Sequential(nn.MaxPool2d(2),blk(64,128))
        self.s2=nn.Sequential(nn.MaxPool2d(2),blk(128,256))
        self.s3=nn.Sequential(nn.MaxPool2d(2),blk(256,512))
        self.depth_head=nn.Sequential(
            nn.Conv2d(512,128,1),nn.BatchNorm2d(128),nn.ReLU(True),
            nn.Conv2d(128,1,1),nn.Sigmoid())
        self.cls_head=nn.Sequential(
            nn.AdaptiveAvgPool2d(1),nn.Flatten(),
            nn.Linear(512,256),nn.BatchNorm1d(256),nn.ReLU(True),
            nn.Dropout(0.5),nn.Linear(256,1))

    def forward(self, x):
        f = self.s3(self.s2(self.s1(self.stem(x))))
        return self.cls_head(f), self.depth_head(f)

# Uji forward pass
dummy = torch.randn(2, 3, 224, 224)
m1, m2 = CDCN(), BaselineCNN()
l1, d1 = m1(dummy)
p1 = sum(p.numel() for p in m1.parameters() if p.requires_grad)
p2 = sum(p.numel() for p in m2.parameters() if p.requires_grad)
print(f"CDCN     : {p1:,} parameter")
print(f"Baseline : {p2:,} parameter")
print(f"Output   : logit={tuple(l1.shape)}, depth={tuple(d1.shape)}")
print("✅ Arsitektur OK!")

---
## 🚀 Bagian 5: Training

### Komponen Utama Training

**Loss Function**
Gabungan dua loss: `L = (1-α)×L_BCE + α×L_MSE` dengan α=0.3
- L_BCE: mengukur kesalahan klasifikasi real/fake
- L_MSE: mengukur kesalahan estimasi depth map

**Metrik Evaluasi Anti-Spoofing**
- **APCER**: % serangan yang lolos sebagai real (bahaya keamanan!)
- **BPCER**: % wajah asli yang ditolak (merugikan pengguna)
- **ACER**: rata-rata APCER dan BPCER — metrik utama yang dioptimasi

**Early Stopping**
Training otomatis berhenti jika ACER validasi tidak membaik
selama 8 epoch berturut-turut — mencegah overfitting.


In [ ]:
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score
from tqdm.notebook import tqdm

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.3, pos_weight=None):
        super().__init__()
        self.alpha = alpha
        self.bce   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.mse   = nn.MSELoss()

    def forward(self, logits, dp, labels, dtgt):
        logits = logits.squeeze(-1)
        if dp.shape[-1] != dtgt.shape[-1]:
            dtgt = F.interpolate(dtgt, dp.shape[2:], mode="bilinear", align_corners=False)
        lc = self.bce(logits, labels)
        ld = self.mse(dp, dtgt)
        return (1-self.alpha)*lc + self.alpha*ld, lc.item(), ld.item()

def calc_metrics(labels, probs, thr=0.5):
    L, P  = np.array(labels), np.array(probs)
    preds = (P >= thr).astype(int)
    acc   = accuracy_score(L, preds)
    try:    auc = roc_auc_score(L, P)
    except: auc = 0.5
    cm = confusion_matrix(L, preds)
    if cm.shape == (2,2):
        tn,fp,fn,tp = cm.ravel()
        apcer = fp/(fp+tn+1e-8); bpcer = fn/(fn+tp+1e-8)
    else: apcer = bpcer = 0.5
    return dict(accuracy=acc*100, auc=auc,
                apcer=apcer*100, bpcer=bpcer*100,
                acer=(apcer+bpcer)/2*100)

def train_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    losses, labels_all, probs_all = [], [], []
    for imgs, labels, depths in tqdm(loader, desc="Train", leave=False):
        imgs=imgs.to(device,non_blocking=True)
        labels=labels.to(device,non_blocking=True)
        depths=depths.to(device,non_blocking=True)
        optimizer.zero_grad()
        if scaler:
            from torch.amp import autocast
            with autocast("cuda"):
                lo, dp = model(imgs); loss,*_ = criterion(lo,dp,labels,depths)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            lo, dp = model(imgs); loss,*_ = criterion(lo,dp,labels,depths)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        losses.append(loss.item())
        probs_all.extend(torch.sigmoid(lo.squeeze(-1)).detach().cpu().numpy().tolist())
        labels_all.extend(labels.cpu().numpy().tolist())
    m = calc_metrics(labels_all, probs_all)
    m["loss"] = np.mean(losses)
    return m

@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    losses, labels_all, probs_all = [], [], []
    for imgs, labels, depths in loader:
        imgs=imgs.to(device,non_blocking=True)
        labels=labels.to(device,non_blocking=True)
        depths=depths.to(device,non_blocking=True)
        lo, dp = model(imgs); loss,*_ = criterion(lo,dp,labels,depths)
        losses.append(loss.item())
        probs_all.extend(torch.sigmoid(lo.squeeze(-1)).cpu().numpy().tolist())
        labels_all.extend(labels.cpu().numpy().tolist())
    m = calc_metrics(labels_all, probs_all)
    m["loss"] = np.mean(losses)
    return m

print("✅ Fungsi training dan metrik siap!")

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/CDCN_AntiSpoofing/"
    import os; os.makedirs(DRIVE_DIR, exist_ok=True)
    DRIVE_OK  = True
    print(f"✅ Drive terhubung: {DRIVE_DIR}")
except:
    DRIVE_OK  = False; DRIVE_DIR = "/content/"
    print("⚠️  Drive tidak tersambung, checkpoint disimpan lokal")

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler

THETA=0.7; LR=5e-4; WD=1e-4; EPOCHS=30; PATIENCE=8

n_r=sum(y_train); n_f=len(y_train)-n_r
pos_w=torch.tensor([n_f/(n_r+1e-8)], dtype=torch.float32).to(DEVICE)
print(f"Class — Real:{n_r} Fake:{n_f} | pos_weight={pos_w.item():.3f}")

cdcn    = CDCN(theta=THETA).to(DEVICE)
opt_c   = AdamW(cdcn.parameters(), lr=LR, weight_decay=WD)
sch_c   = CosineAnnealingLR(opt_c, T_max=EPOCHS, eta_min=1e-6)
crit    = CombinedLoss(alpha=0.3, pos_weight=pos_w).to(DEVICE)
scaler  = GradScaler("cuda") if DEVICE.type=="cuda" else None
hist_c  = {"train":[], "val":[]}
best_acer, best_w, no_imp = float("inf"), None, 0

print("="*68)
print(f"🚀 TRAINING CDCN  θ={THETA}  LR={LR}")
print("="*68)
print(f"{'Ep':>4}|{'TrLoss':>9}|{'TrAcc':>8}|{'VaLoss':>9}|{'VaAcc':>8}|{'ACER':>8}|{'AUC':>8}")
print("-"*68)

for ep in range(1, EPOCHS+1):
    tm = train_epoch(cdcn, train_loader, opt_c, crit, DEVICE, scaler)
    vm = val_epoch(cdcn, val_loader, crit, DEVICE)
    sch_c.step()
    hist_c["train"].append(tm); hist_c["val"].append(vm)
    print(f"{ep:>4}|{tm['loss']:>9.4f}|{tm['accuracy']:>7.2f}%|"
          f"{vm['loss']:>9.4f}|{vm['accuracy']:>7.2f}%|"
          f"{vm['acer']:>7.2f}%|{vm['auc']:>8.4f}")
    if vm["acer"] < best_acer:
        best_acer=vm["acer"]
        best_w={k:v.clone() for k,v in cdcn.state_dict().items()}
        ckpt={"model_state_dict":best_w,"theta":THETA,"epoch":ep,"acer":best_acer}
        torch.save(ckpt,"/content/cdcn_best.pth")
        if DRIVE_OK: torch.save(ckpt, f"{DRIVE_DIR}cdcn_best.pth")
        no_imp=0; print(f"     ✨ Best ACER={best_acer:.2f}% (Ep{ep})")
    else:
        no_imp+=1
    if no_imp >= PATIENCE:
        print(f"\n⏹️  Early stopping ep{ep}"); break

cdcn.load_state_dict(best_w)
print(f"\n✅ CDCN selesai! Best ACER val: {best_acer:.2f}%")

In [ ]:
baseline=BaselineCNN().to(DEVICE)
opt_b=AdamW(baseline.parameters(),lr=LR,weight_decay=WD)
sch_b=CosineAnnealingLR(opt_b,T_max=EPOCHS,eta_min=1e-6)
scaler_b=GradScaler("cuda") if DEVICE.type=="cuda" else None
hist_b={"train":[],"val":[]}
best_acer_b,best_w_b,no_imp_b=float("inf"),None,0

print("="*68)
print(f"🔵 TRAINING BASELINE CNN")
print("="*68)
print(f"{'Ep':>4}|{'TrLoss':>9}|{'TrAcc':>8}|{'VaLoss':>9}|{'VaAcc':>8}|{'ACER':>8}|{'AUC':>8}")
print("-"*68)

for ep in range(1,EPOCHS+1):
    tm=train_epoch(baseline,train_loader,opt_b,crit,DEVICE,scaler_b)
    vm=val_epoch(baseline,val_loader,crit,DEVICE)
    sch_b.step()
    hist_b["train"].append(tm); hist_b["val"].append(vm)
    print(f"{ep:>4}|{tm['loss']:>9.4f}|{tm['accuracy']:>7.2f}%|"
          f"{vm['loss']:>9.4f}|{vm['accuracy']:>7.2f}%|"
          f"{vm['acer']:>7.2f}%|{vm['auc']:>8.4f}")
    if vm["acer"]<best_acer_b:
        best_acer_b=vm["acer"]
        best_w_b={k:v.clone() for k,v in baseline.state_dict().items()}
        no_imp_b=0; print(f"     ✨ Best ACER={best_acer_b:.2f}% (Ep{ep})")
    else: no_imp_b+=1
    if no_imp_b>=PATIENCE: print(f"\n⏹️  Early stopping ep{ep}"); break

baseline.load_state_dict(best_w_b)
print(f"\n✅ Baseline selesai! Best ACER val: {best_acer_b:.2f}%")

---
## 📊 Bagian 6: Visualisasi Kurva Training

### Cara Membaca Kurva

| Kurva | Tanda Sehat | Tanda Bermasalah |
|---|---|---|
| Loss ↓ | Training & val sama-sama turun | Val naik saat training turun (overfit) |
| Accuracy ↑ | Keduanya naik konsisten | Val stagnan saat training terus naik |
| Gap train-val | Kecil dan stabil | Membesar seiring epoch (overfit) |

> **Catatan:** Di sini kita belum bisa menilai overfit secara definitif
> karena kita hanya melihat val set. Evaluasi overfit yang sesungguhnya
> menggunakan test set ada di Bagian 8.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.dpi"] = 110

def get_m(hist, split, key):
    return [e[key] for e in hist[split]]

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

configs = [
    ("loss",     "Loss",        False),
    ("accuracy", "Accuracy (%)", True),
    ("acer",     "ACER (%)",    False),
    ("auc",      "AUC",         True),
]
for ax, (key, ylabel, up) in zip([axes[0,0],axes[0,1],axes[0,2],axes[1,0]], configs):
    tc=get_m(hist_c,"train",key); vc=get_m(hist_c,"val",key)
    tb=get_m(hist_b,"train",key); vb=get_m(hist_b,"val",key)
    ec=range(1,len(tc)+1); eb=range(1,len(tb)+1)
    ax.plot(ec,tc,"--",color="#27AE60",lw=1.5,alpha=0.7,label="CDCN Train")
    ax.plot(ec,vc,"-", color="#27AE60",lw=2.5,label="CDCN Val")
    ax.plot(eb,tb,"--",color="#2980B9",lw=1.5,alpha=0.7,label="Baseline Train")
    ax.plot(eb,vb,"-", color="#2980B9",lw=2.5,label="Baseline Val")
    ax.fill_between(ec,tc,vc,alpha=0.08,color="#E74C3C")
    ax.set_title(f"{ylabel} ({'↑' if up else '↓'})",fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

best_idx   = int(np.argmin([e["acer"] for e in hist_c["val"]]))
best_idx_b = int(np.argmin([e["acer"] for e in hist_b["val"]]))

ax = axes[1,1]
mnames=["Accuracy","AUC×100","ACER"]
cv=[hist_c["val"][best_idx]["accuracy"],hist_c["val"][best_idx]["auc"]*100,hist_c["val"][best_idx]["acer"]]
bv=[hist_b["val"][best_idx_b]["accuracy"],hist_b["val"][best_idx_b]["auc"]*100,hist_b["val"][best_idx_b]["acer"]]
x=np.arange(3); w=0.35
b1=ax.bar(x-w/2,cv,w,label=f"CDCN ep{best_idx+1}",color="#27AE60",alpha=0.85)
b2=ax.bar(x+w/2,bv,w,label=f"Baseline ep{best_idx_b+1}",color="#2980B9",alpha=0.85)
for bars in [b1,b2]:
    for bar in bars:
        h=bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2,h+0.5,f"{h:.1f}",ha="center",fontsize=8,fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(mnames)
ax.set_title("Perbandingan di Best Checkpoint\n(Val Set — BUKAN test set)",fontweight="bold")
ax.legend(fontsize=8); ax.grid(axis="y",alpha=0.3)

ax=axes[1,2]; ax.axis("off")
tr_a=hist_c["train"][best_idx]["acer"]; va_a=hist_c["val"][best_idx]["acer"]
summary=(f"RINGKASAN TRAINING\n{'─'*28}\n\nCDCN\n"
         f"  Best epoch    : {best_idx+1}\n"
         f"  Train ACER    : {tr_a:.2f}%\n"
         f"  Val   ACER    : {va_a:.2f}%\n"
         f"  Gap (val-tr)  : {va_a-tr_a:+.2f}%\n\n"
         f"Baseline\n  Best epoch    : {best_idx_b+1}\n"
         f"  Val   ACER    : {hist_b['val'][best_idx_b]['acer']:.2f}%\n\n"
         f"{'─'*28}\n\n⚠️  Ini hanya val set!\nEvaluasi final di\nBagian 7 (test set).")
ax.text(0.05,0.97,summary,transform=ax.transAxes,fontsize=9.5,
        verticalalignment="top",fontfamily="monospace",
        bbox=dict(boxstyle="round,pad=0.5",facecolor="#F8F9FA",edgecolor="#CCCCCC"))

plt.suptitle("Kurva Training: CDCN vs Baseline CNN",fontsize=13,fontweight="bold")
plt.tight_layout()
plt.savefig("/content/training_curves.png",bbox_inches="tight",dpi=120)
plt.show()
print("✅ Disimpan: /content/training_curves.png")

---
## 🏆 Bagian 7: Evaluasi pada Test Set

### Mengapa Test Set Digunakan Terakhir?
Test set adalah data yang **belum pernah dilihat model** — tidak
untuk training, tidak untuk memilih checkpoint (itu tugas val set).

Test set digunakan **SATU KALI** di akhir. Jika digunakan berkali-kali
untuk tuning, maka model sudah "melihat" test set secara tidak langsung
dan angka evaluasinya tidak lagi jujur.

### Kalibrasi Threshold (Youden's J)
Default threshold 0.5 belum tentu optimal. Kita kalibrasi menggunakan
Youden's J pada VAL SET (bukan test set!):

```
threshold* = argmax(TPR - FPR)
```

Kalibrasi dari val set mencari threshold yang menyeimbangkan
true positive rate dan false positive rate secara optimal.


In [ ]:
import torch, numpy as np
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, precision_recall_curve, auc)
import matplotlib.pyplot as plt, seaborn as sns

@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    all_labels, all_probs = [], []
    for imgs, labels, _ in loader:
        lo, _ = model(imgs.to(device))
        all_probs.extend(torch.sigmoid(lo.squeeze(-1)).cpu().numpy().tolist())
        all_labels.extend(labels.numpy().tolist())
    return np.array(all_labels), np.array(all_probs)

# Kalibrasi dari VAL SET
@torch.no_grad()
def find_threshold(model, loader, device):
    labels, probs = predict_all(model, loader, device)
    fpr, tpr, thresholds = roc_curve(labels, probs)
    j      = tpr - fpr
    best   = int(np.argmax(j))
    return float(thresholds[best]), float(tpr[best]), float(fpr[best])

print("🔧 Kalibrasi threshold dari VAL SET (bukan test set)...")
OPTIMAL_THR, opt_tpr, opt_fpr = find_threshold(cdcn, val_loader, DEVICE)
print(f"   Optimal : {OPTIMAL_THR:.4f} (TPR={opt_tpr:.3f}, FPR={opt_fpr:.3f})")
print(f"   Default : 0.5000")

print("\n📊 Mengevaluasi di TEST SET (pertama kali model melihat data ini)...")
y_true, p_cdcn = predict_all(cdcn,     test_loader, DEVICE)
_,      p_base  = predict_all(baseline, test_loader, DEVICE)

mc_def = calc_metrics(y_true, p_cdcn, thr=0.5)
mc_opt = calc_metrics(y_true, p_cdcn, thr=OPTIMAL_THR)
mb     = calc_metrics(y_true, p_base, thr=0.5)

print("\n"+"="*65)
print("🏆 HASIL EVALUASI — TEST SET")
print("="*65)
print(f"{'Metrik':<13}|{'CDCN(thr=0.5)':>15}|{'CDCN(opt_thr)':>15}|{'Baseline':>11}")
print("-"*65)
for key,name in [("accuracy","Accuracy(%)"),("auc","AUC"),
                  ("apcer","APCER(%)"),("bpcer","BPCER(%)"),("acer","ACER(%)")]:
    vd,vo,vb=mc_def[key],mc_opt[key],mb[key]
    fmt=".4f" if key=="auc" else ".2f"
    print(f"{name:<13}|{vd:>15{fmt}}|{vo:>15{fmt}}|{vb:>11{fmt}}")
print("="*65)

In [ ]:
# Visualisasi evaluasi test set
fig, axes = plt.subplots(2, 3, figsize=(17,11))
pd_c=(p_cdcn>=OPTIMAL_THR).astype(int); pd_b=(p_base>=0.5).astype(int)

ax=axes[0,0]
for nm,yp,c in [("CDCN",p_cdcn,"#E74C3C"),("Baseline",p_base,"#3498DB")]:
    fp_,tp_,_=roc_curve(y_true,yp)
    ax.plot(fp_,tp_,color=c,lw=2.5,label=f"{nm} AUC={auc(fp_,tp_):.4f}")
ax.plot([0,1],[0,1],"k--",lw=1,alpha=0.4)
ax.set_title("ROC Curve\n(pojok kiri atas = sempurna)",fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")

ax=axes[0,1]
sns.heatmap(confusion_matrix(y_true,pd_c),annot=True,fmt="d",cmap="Greens",ax=ax,
            xticklabels=["Fake","Real"],yticklabels=["Fake","Real"],
            linewidths=0.5,annot_kws={"size":14,"weight":"bold"})
ax.set_title("Confusion Matrix — CDCN\n(diagonal = prediksi benar)",fontweight="bold")
ax.set_xlabel("Prediksi"); ax.set_ylabel("Label Sebenarnya")

ax=axes[0,2]
sns.heatmap(confusion_matrix(y_true,pd_b),annot=True,fmt="d",cmap="Blues",ax=ax,
            xticklabels=["Fake","Real"],yticklabels=["Fake","Real"],
            linewidths=0.5,annot_kws={"size":14,"weight":"bold"})
ax.set_title("Confusion Matrix — Baseline",fontweight="bold")
ax.set_xlabel("Prediksi"); ax.set_ylabel("Label Sebenarnya")

ax=axes[1,0]
for nm,yp,c in [("CDCN",p_cdcn,"#E74C3C"),("Baseline",p_base,"#3498DB")]:
    p,r,_=precision_recall_curve(y_true,yp)
    ax.plot(r,p,color=c,lw=2.5,label=f"{nm} PR-AUC={auc(r,p):.4f}")
ax.set_title("Precision-Recall Curve",fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)

ax=axes[1,1]
for lv,ln,c in [(0,"Fake","#E74C3C"),(1,"Real","#2ECC71")]:
    ax.hist(p_cdcn[y_true==lv],bins=30,alpha=0.6,color=c,label=ln,density=True)
ax.axvline(OPTIMAL_THR,color="navy",ls="--",lw=2,label=f"opt_thr={OPTIMAL_THR:.3f}")
ax.axvline(0.5,color="gray",ls=":",lw=1.5,label="default=0.5")
ax.set_title("Distribusi Skor — CDCN\n(tidak overlap = model bagus)",fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax=axes[1,2]
mks=["accuracy","auc","apcer","bpcer","acer"]
mns=["Acc(%)","AUC×100","APCER","BPCER","ACER"]
cv=[mc_opt["accuracy"],mc_opt["auc"]*100,mc_opt["apcer"],mc_opt["bpcer"],mc_opt["acer"]]
bv=[mb["accuracy"],mb["auc"]*100,mb["apcer"],mb["bpcer"],mb["acer"]]
x=np.arange(5); w=0.35
b1=ax.bar(x-w/2,cv,w,label="CDCN",color="#E74C3C",alpha=0.85)
b2=ax.bar(x+w/2,bv,w,label="Baseline",color="#3498DB",alpha=0.85)
for bars in [b1,b2]:
    for bar in bars:
        h=bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2,h+0.3,f"{h:.1f}",ha="center",fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(mns,rotation=10)
ax.set_title("Perbandingan Metrik — Test Set",fontweight="bold")
ax.legend(fontsize=8); ax.grid(axis="y",alpha=0.3)

plt.suptitle("Evaluasi Komprehensif pada Test Set",fontsize=13,fontweight="bold")
plt.tight_layout()
plt.savefig("/content/eval_results.png",bbox_inches="tight",dpi=120)
plt.show()

print("\n📋 Classification Report — CDCN")
print(classification_report(y_true,pd_c,target_names=["Fake","Real"]))

### 📌 Interpretasi Hasil Test Set

**Confusion Matrix — cara membacanya:**
```
                Prediksi Fake  |  Prediksi Real
Actual Fake  |  TN (✅ benar)  |  FP (❌ → APCER, serangan lolos!)
Actual Real  |  FN (❌ → BPCER, user ditolak) | TP (✅ benar)
```

**Score Distribution:**
- Ideal: distribusi Real dan Fake tidak overlap sama sekali
- Ada overlap → model kesulitan di area ambigus
- Threshold optimal memisahkan dua distribusi dengan error minimal

**Classification Report:**
- Precision=1.00 dan Recall=1.00 di dataset kecil dan homogen ini
  perlu diinterpretasikan hati-hati — bisa jadi karena dataset terlalu mudah
- Lihat Bagian 8 untuk analisis apakah ini tanda overfitting


---
## 🔍 Bagian 8: Evaluasi Overfitting / Underfitting

### Konsep yang Benar

Setelah mendapat performa test set, kita bandingkan dengan
performa di data training:

| Kondisi | Train Acc | Test Acc | Makna |
|---|---|---|---|
| Underfitting | Rendah | Rendah | Belum cukup belajar |
| Good Fit | Tinggi | Tinggi ≈ Train | Generalisasi baik |
| Mild Overfit | Tinggi | Sedikit lebih rendah | Wajar untuk dataset kecil |
| Overfitting | Sangat tinggi | Jauh lebih rendah | Hafal data training |

### Mengapa Ini Penting untuk Anti-Spoofing?
Model yang overfit "hafal" kondisi kamera, pencahayaan, dan wajah
tertentu dari dataset training. Di dunia nyata dengan kondisi berbeda,
model akan gagal — inilah yang menjelaskan hasil tidak konsisten di live camera.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

best_idx   = int(np.argmin([e["acer"] for e in hist_c["val"]]))
best_idx_b = int(np.argmin([e["acer"] for e in hist_b["val"]]))

# Performa training vs test
train_acc_c  = hist_c["train"][best_idx]["accuracy"]
train_acer_c = hist_c["train"][best_idx]["acer"]
train_auc_c  = hist_c["train"][best_idx]["auc"]
test_acc_c   = mc_opt["accuracy"]
test_acer_c  = mc_opt["acer"]
test_auc_c   = mc_opt["auc"]

train_acc_b  = hist_b["train"][best_idx_b]["accuracy"]
train_acer_b = hist_b["train"][best_idx_b]["acer"]
test_acc_b   = mb["accuracy"]
test_acer_b  = mb["acer"]

gap_acc_c  = train_acc_c  - test_acc_c
gap_acer_c = test_acer_c  - train_acer_c
gap_acc_b  = train_acc_b  - test_acc_b

def get_verdict(gap_acc, train_acc):
    if train_acc < 80:
        return "UNDERFITTING ❌","#E67E22","Model belum cukup belajar dari data training."
    elif gap_acc > 10:
        return "OVERFITTING ❌","#E74C3C","Model hafal training tapi buruk di data baru."
    elif gap_acc > 3:
        return "MILD OVERFIT ⚠️ ","#F39C12","Ada gap kecil. Wajar untuk dataset kecil."
    else:
        return "GOOD FIT ✅","#2ECC71","Performa training dan test konsisten."

status_c,color_c,advice_c = get_verdict(gap_acc_c, train_acc_c)
status_b,color_b,advice_b = get_verdict(gap_acc_b, train_acc_b)

print("="*60)
print("  EVALUASI OVERFITTING — TRAINING vs TEST SET")
print("="*60)
print(f"  Best epoch CDCN     : {best_idx+1}")
print(f"  Best epoch Baseline : {best_idx_b+1}")
print()
print(f"  {'Metrik':<16} {'Train':>9} {'Test':>9} {'Gap':>9}")
print(f"  {'-'*46}")
print(f"  {'Accuracy (%)':<16} {train_acc_c:>9.2f} {test_acc_c:>9.2f} {gap_acc_c:>+9.2f}")
print(f"  {'ACER (%)':<16} {train_acer_c:>9.2f} {test_acer_c:>9.2f} {gap_acer_c:>+9.2f}")
print(f"  {'AUC':<16} {train_auc_c:>9.4f} {test_auc_c:>9.4f} {test_auc_c-train_auc_c:>+9.4f}")
print(f"  {'-'*46}")
print(f"  Gap Accuracy positif = train lebih baik dari test (overfit)")
print()
print(f"  CDCN     → {status_c}")
print(f"             {advice_c}")
print(f"\n  Baseline → {status_b}")
print(f"             {advice_b}")
print("="*60)

fig, axes = plt.subplots(1, 3, figsize=(16,5))
models=["CDCN","Baseline"]; x=np.arange(2); w=0.35

ax=axes[0]
b1=ax.bar(x-w/2,[train_acc_c,train_acc_b],w,label="Training",color="#3498DB",alpha=0.85)
b2=ax.bar(x+w/2,[test_acc_c, test_acc_b], w,label="Test",    color="#E74C3C",alpha=0.85)
for bars in [b1,b2]:
    for bar in bars:
        h=bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2,h+0.3,f"{h:.2f}%",ha="center",fontsize=9,fontweight="bold")
for i,(tr,te) in enumerate(zip([train_acc_c,train_acc_b],[test_acc_c,test_acc_b])):
    gap=tr-te; col="#E74C3C" if gap>3 else "#27AE60"
    ax.annotate(f"Gap:{gap:+.2f}%",xy=(i,min(tr,te)-2),ha="center",fontsize=9,color=col,fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_ylim(max(0,min(train_acc_c,test_acc_c,train_acc_b,test_acc_b)-10),105)
ax.set_title("Accuracy: Training vs Test\n(gap kecil = good fit)",fontweight="bold")
ax.set_ylabel("Accuracy (%)"); ax.legend(); ax.grid(axis="y",alpha=0.3)

ax=axes[1]
b1=ax.bar(x-w/2,[train_acer_c,train_acer_b],w,label="Training",color="#3498DB",alpha=0.85)
b2=ax.bar(x+w/2,[test_acer_c, test_acer_b], w,label="Test",    color="#E74C3C",alpha=0.85)
for bars in [b1,b2]:
    for bar in bars:
        h=bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2,h+0.1,f"{h:.2f}%",ha="center",fontsize=9,fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_title("ACER: Training vs Test\n(↓ lebih baik, gap kecil = good fit)",fontweight="bold")
ax.set_ylabel("ACER (%)"); ax.legend(); ax.grid(axis="y",alpha=0.3)

ax=axes[2]; ax.axis("off")
txt=(f"VERDICT EVALUASI MODEL\n{'─'*28}\n\n"
     f"CDCN (Epoch {best_idx+1})\n"
     f"  Train Acc  : {train_acc_c:.2f}%\n"
     f"  Test  Acc  : {test_acc_c:.2f}%\n"
     f"  Gap        : {gap_acc_c:+.2f}%\n\n"
     f"  Train ACER : {train_acer_c:.2f}%\n"
     f"  Test  ACER : {test_acer_c:.2f}%\n\n"
     f"  Status: {status_c}\n\n"
     f"{'─'*28}\n\n"
     f"Baseline (Epoch {best_idx_b+1})\n"
     f"  Train Acc  : {train_acc_b:.2f}%\n"
     f"  Test  Acc  : {test_acc_b:.2f}%\n"
     f"  Gap        : {gap_acc_b:+.2f}%\n\n"
     f"  Status: {status_b}\n\n"
     f"{'─'*28}\n\n{advice_c}")
ax.text(0.05,0.97,txt,transform=ax.transAxes,fontsize=9,
        verticalalignment="top",fontfamily="monospace",
        bbox=dict(boxstyle="round,pad=0.5",facecolor="#F8F9FA",
                  edgecolor=color_c,linewidth=2.5))

plt.suptitle("Evaluasi Overfitting / Underfitting\nTraining vs Test Set",
             fontsize=13,fontweight="bold")
plt.tight_layout()
plt.savefig("/content/overfit_eval.png",bbox_inches="tight",dpi=120)
plt.show()
print("✅ Disimpan: /content/overfit_eval.png")

### 📌 Interpretasi Overfitting Analysis

**Gap Accuracy (Train − Test):**
- < 3%  → Good Fit — model belajar fitur yang bisa digeneralisasi
- 3–10% → Mild Overfit — wajar untuk dataset kecil ini
- > 10% → Overfitting serius — model hafal data training

**Penting untuk diingat:**
Bahkan jika hasilnya terlihat "Good Fit" di sini, generalisasi sejati
hanya bisa dibuktikan melalui **cross-dataset testing** — melatih di
dataset ini lalu menguji di dataset yang sama sekali berbeda seperti
Replay-Attack (Idiap) atau OULU-NPU.


---
## 🔬 Bagian 9: Ablation Study — Pengaruh Nilai θ

### Apa itu Ablation Study?
Ablation study mengukur kontribusi komponen tertentu dengan
memvariasikannya dan mengamati dampaknya.

Di sini kita memvariasikan nilai θ pada CDC layer untuk memvalidasi
apakah θ=0.7 (rekomendasi paper) memang optimal.

**Ekspektasi:**
- θ=0.0 (vanilla) seharusnya lebih buruk — membuktikan CDC berkontribusi
- θ=0.7 seharusnya optimal atau mendekati optimal
- θ=1.0 (pure gradient) seharusnya lebih buruk dari 0.7


In [ ]:
from torch.optim import AdamW
from torch.amp import GradScaler

THETA_VALS = [0.0, 0.3, 0.5, 0.7, 1.0]
ABL_EPOCHS = 5
abl_results = []

print(f"🔬 Ablation Study θ ({ABL_EPOCHS} epoch per nilai)")
print("="*55)

for th in THETA_VALS:
    m   = CDCN(theta=th).to(DEVICE)
    opt = AdamW(m.parameters(), lr=5e-4, weight_decay=1e-4)
    sc  = GradScaler("cuda") if DEVICE.type=="cuda" else None
    for _ in range(ABL_EPOCHS):
        train_epoch(m, train_loader, opt, crit, DEVICE, sc)
    vm  = val_epoch(m, val_loader, crit, DEVICE)
    mode= "Vanilla" if th==0 else ("PureGrad" if th==1 else "CDC")
    abl_results.append({"theta":th,"mode":mode,**vm})
    print(f"  θ={th} ({mode:<9}) Acc={vm['accuracy']:.2f}%  ACER={vm['acer']:.2f}%  AUC={vm['auc']:.4f}")

print("="*55)

fig, axes = plt.subplots(1, 3, figsize=(15,5))
thetas = [r["theta"] for r in abl_results]
cols   = ["#E74C3C" if r["theta"]==0.7 else "#3498DB" for r in abl_results]

for ax,(key,label,better) in zip(axes,[
    ("acer","ACER (%)","↓"),("accuracy","Accuracy (%)","↑"),("auc","AUC","↑")]):
    vals = [r[key] for r in abl_results]
    bars = ax.bar([str(t) for t in thetas], vals, color=cols, alpha=0.85, edgecolor="white")
    for bar,val in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.2,
                f"{val:.2f}",ha="center",fontsize=9,fontweight="bold")
    ax.axvline(3,color="#E74C3C",ls="--",lw=2,label="θ=0.7 (paper)")
    ax.set_title(f"{label} ({better} lebih baik)",fontweight="bold")
    ax.set_xlabel("θ"); ax.set_ylabel(label)
    ax.legend(fontsize=8); ax.grid(axis="y",alpha=0.3)

plt.suptitle("Ablation Study: Pengaruh Nilai θ pada CDC Layer\n"
             "(batang merah = θ=0.7, nilai dari paper asli)",
             fontsize=13,fontweight="bold")
plt.tight_layout()
plt.savefig("/content/ablation.png",bbox_inches="tight",dpi=120)
plt.show()

best_abl = min(abl_results, key=lambda x: x["acer"])
print(f"\n📌 θ dengan ACER terbaik: θ={best_abl['theta']} (ACER={best_abl['acer']:.2f}%)")

---
## 🎥 Bagian 10: Inference Real-Time

### Inference vs Evaluasi
- **Evaluasi**: data berlabel → menghasilkan metrik (accuracy, ACER, AUC)
- **Inference**: data baru tanpa label → menghasilkan prediksi langsung

### Tiga Skenario
1. **Wajah asli di webcam** → ekspektasi: LIVE ✅
2. **Layar HP di webcam** (replay attack) → ekspektasi: SPOOF ✅
3. **Upload gambar** → prediksi per gambar

### Cara Membaca Output
- **Kotak hijau + LIVE**: wajah nyata terdeteksi
- **Kotak merah + SPOOF**: serangan terdeteksi
- **Depth map** (pojok kanan bawah):
  - Warna panas (merah/oranye) = ada struktur kedalaman 3D = wajah asli
  - Warna dingin (biru) = tidak ada kedalaman = layar/foto datar


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cdcn   = CDCN(theta=THETA).to(DEVICE)
model_path = f"{DRIVE_DIR}cdcn_best.pth" if DRIVE_OK else "/content/cdcn_best.pth"
ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
cdcn.load_state_dict(ckpt["model_state_dict"])
cdcn.eval()
print(f"✅ Model loaded! Best ACER: {ckpt['acer']:.2f}%")

In [ ]:
import torch, cv2, numpy as np, io
from base64 import b64decode
from IPython.display import display, Image
from google.colab.output import eval_js
import PIL.Image

_MEAN = np.array([0.485,0.456,0.406],dtype=np.float32)
_STD  = np.array([0.229,0.224,0.225],dtype=np.float32)

try:    _THR = float(OPTIMAL_THR); print(f"✅ Threshold optimal: {_THR:.4f}")
except: _THR = 0.5; print("⚠️  Pakai default 0.5")

def preprocess(bgr):
    rgb = cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb,(224,224)).astype(np.float32)/255.0
    rgb = (rgb-_MEAN)/_STD
    return torch.from_numpy(rgb.transpose(2,0,1)).float().unsqueeze(0)

@torch.no_grad()
def predict_frame(model, bgr, device, thr=None):
    if thr is None: thr = _THR
    model.eval()
    t = preprocess(bgr).to(device)
    lo, dp = model(t)
    prob  = torch.sigmoid(lo.squeeze()).item()
    label = "REAL" if prob >= thr else "FAKE"
    h,w   = bgr.shape[:2]; color=(0,210,0) if label=="REAL" else (0,0,210)
    out   = bgr.copy()
    cv2.rectangle(out,(0,0),(w,55),(0,0,0),-1)
    cv2.addWeighted(out,0.5,bgr,0.5,0,out)
    cv2.putText(out,f"{'LIVE' if label=='REAL' else 'SPOOF'}  {prob:.1%}",
                (10,40),cv2.FONT_HERSHEY_SIMPLEX,1.1,color,3,cv2.LINE_AA)
    cv2.rectangle(out,(0,0),(w-1,h-1),color,5)
    dm=cv2.resize(dp.squeeze().cpu().numpy(),(110,80))
    dm=cv2.applyColorMap((dm*255).astype(np.uint8),cv2.COLORMAP_JET)
    if h>90 and w>120:
        out[h-85:h-5,w-115:w-5]=dm
        cv2.putText(out,"depth",(w-115,h-90),cv2.FONT_HERSHEY_SIMPLEX,0.35,(255,255,255),1)
    return label,prob,out

def capture_frame():
    data_url=eval_js("""
    (async()=>{
        let v=document.querySelector('#_fas_v');
        if(!v){
            v=document.createElement('video');
            v.id='_fas_v';v.setAttribute('autoplay','');v.setAttribute('playsinline','');
            v.style.cssText='width:300px;border:3px solid #2ecc71;border-radius:8px;display:block;';
            const a=document.querySelector('#output-area')||document.body;
            a.prepend(v);
            const s=await navigator.mediaDevices.getUserMedia({video:{width:640,height:480,facingMode:'user'}});
            v.srcObject=s;
            await new Promise(r=>{v.onloadedmetadata=r;});
            await v.play();
            await new Promise(r=>setTimeout(r,800));
        }
        const c=document.createElement('canvas');
        c.width=v.videoWidth||640;c.height=v.videoHeight||480;
        c.getContext('2d').drawImage(v,0,0);
        return c.toDataURL('image/jpeg',0.85);
    })()
    """)
    if not data_url or not data_url.startswith("data:"):
        raise RuntimeError("Gagal capture — izinkan kamera!")
    raw=b64decode(data_url.split(",")[1])
    return cv2.cvtColor(np.array(PIL.Image.open(io.BytesIO(raw))),cv2.COLOR_RGB2BGR)

def to_img(bgr):
    _,buf=cv2.imencode(".jpg",bgr); return Image(data=buf.tobytes())

print("✅ Inference helper siap! Jalankan cell berikutnya.")

In [ ]:
# ── Skenario 1: Webcam Wajah Asli ────────────────────────────
from IPython.display import clear_output
import time, numpy as np

print("🟢 SKENARIO 1: Wajah Asli — arahkan wajah Anda ke kamera")
print("⏹️  Klik STOP untuk menghentikan\n")

log1, fps1, err = [], [], 0
try:
    for i in range(100):
        t0=time.time()
        try: frame=capture_frame(); err=0
        except Exception as e:
            err+=1
            if err>=5: print(f"❌ {e}"); break
            time.sleep(1); continue
        label,prob,ann=predict_frame(cdcn,frame,DEVICE)
        fps=1/max(time.time()-t0,1e-6); fps1.append(fps)
        avg=float(np.mean(fps1[-10:]))
        cv2.putText(ann,f"Skenario 1: Wajah Asli | FPS:{avg:.1f} | {i+1}/100",
                    (10,ann.shape[0]-12),cv2.FONT_HERSHEY_SIMPLEX,0.45,(200,200,200),1)
        log1.append({"label":label,"prob":prob})
        if i%2==0:
            clear_output(wait=True)
            print(f"{'🟢 REAL' if label=='REAL' else '🔴 FAKE'} | prob={prob:.3f} | FPS={avg:.1f}")
            display(to_img(ann))
except KeyboardInterrupt: print("\n⏹️  Dihentikan.")
except Exception as e: print(f"\n❌ {e}")

if log1:
    n=len(log1); rc=sum(1 for x in log1 if x["label"]=="REAL")
    print(f"\n📊 Skenario 1 — {n} frame | 🟢 {rc}({rc/n*100:.0f}%) | 🔴 {n-rc}({(n-rc)/n*100:.0f}%)")
    print(f"   Avg prob: {np.mean([x['prob'] for x in log1]):.3f} | Ekspektasi: mayoritas REAL")

In [ ]:
# ── Skenario 2: Webcam Replay Attack ─────────────────────────
from IPython.display import clear_output
import time, numpy as np

print("🔴 SKENARIO 2: Replay Attack — arahkan LAYAR HP ke kamera")
print("⏹️  Klik STOP untuk menghentikan\n")

log2, fps2, err = [], [], 0
try:
    for i in range(100):
        t0=time.time()
        try: frame=capture_frame(); err=0
        except Exception as e:
            err+=1
            if err>=5: print(f"❌ {e}"); break
            time.sleep(1); continue
        label,prob,ann=predict_frame(cdcn,frame,DEVICE)
        fps=1/max(time.time()-t0,1e-6); fps2.append(fps)
        avg=float(np.mean(fps2[-10:]))
        cv2.putText(ann,f"Skenario 2: Replay Attack | FPS:{avg:.1f} | {i+1}/100",
                    (10,ann.shape[0]-12),cv2.FONT_HERSHEY_SIMPLEX,0.45,(200,200,200),1)
        log2.append({"label":label,"prob":prob})
        if i%2==0:
            clear_output(wait=True)
            print(f"{'🟢 REAL' if label=='REAL' else '🔴 FAKE'} | prob={prob:.3f} | FPS={avg:.1f}")
            display(to_img(ann))
except KeyboardInterrupt: print("\n⏹️  Dihentikan.")
except Exception as e: print(f"\n❌ {e}")

if log2:
    n=len(log2); fc=sum(1 for x in log2 if x["label"]=="FAKE")
    print(f"\n📊 Skenario 2 — {n} frame | 🟢 {n-fc}({(n-fc)/n*100:.0f}%) | 🔴 {fc}({fc/n*100:.0f}%)")
    print(f"   Avg prob: {np.mean([x['prob'] for x in log2]):.3f} | Ekspektasi: mayoritas FAKE")

In [ ]:
# ── Skenario 3: Upload Gambar ────────────────────────────────
from google.colab import files
import matplotlib.pyplot as plt

print("📁 SKENARIO 3: Upload gambar (foto asli ATAU screenshot HP)")
ups = files.upload()

if ups:
    n_f = len(ups)
    fig, axes = plt.subplots(1, n_f, figsize=(6*n_f, 6))
    if n_f == 1: axes = [axes]
    results = []
    for ax, (fname, fdata) in zip(axes, ups.items()):
        arr   = np.frombuffer(fdata, np.uint8)
        frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if frame is None: ax.text(0.5,0.5,"Gagal",ha="center"); ax.axis("off"); continue
        # Resize sesuai pipeline training
        frame_r = cv2.resize(frame, (224,224))
        label, prob, ann = predict_frame(cdcn, frame_r, DEVICE)
        results.append({"file":fname,"label":label,"prob":prob})
        # Overlay pada gambar asli
        disp = frame.copy(); h,w=disp.shape[:2]
        color=(0,210,0) if label=="REAL" else (0,0,210)
        cv2.rectangle(disp,(0,0),(w,50),(0,0,0),-1)
        cv2.addWeighted(disp,0.5,frame.copy(),0.5,0,disp)
        cv2.putText(disp,f"{'LIVE' if label=='REAL' else 'SPOOF'}  {prob:.1%}",
                    (10,36),cv2.FONT_HERSHEY_SIMPLEX,1.0,color,2,cv2.LINE_AA)
        cv2.rectangle(disp,(0,0),(w-1,h-1),color,4)
        col_hex="#2ECC71" if label=="REAL" else "#E74C3C"
        icon="🟢" if label=="REAL" else "🔴"
        ax.imshow(cv2.cvtColor(disp,cv2.COLOR_BGR2RGB))
        ax.set_title(f"{icon} {label}\nProb: {prob:.1%}\n{fname[:20]}",
                     fontweight="bold",color=col_hex)
        ax.axis("off")
    plt.suptitle("Skenario 3: Inference Upload Gambar",fontsize=13,fontweight="bold")
    plt.tight_layout()
    plt.savefig("/content/inference_upload.png",bbox_inches="tight",dpi=120)
    plt.show()
    print(f"\n{'='*45}")
    for res in results:
        icon="🟢" if res["label"]=="REAL" else "🔴"
        print(f"  {res['file'][:25]:<25} {icon} {res['label']:<8} {res['prob']:.1%}")
    print(f"  Threshold: {_THR:.4f}")

---
## 💾 Bagian 11: Simpan & Download


In [ ]:
import os, shutil, json, subprocess
from google.colab import files

OUT = "/content/output_cdcn"
os.makedirs(OUT, exist_ok=True)

# Simpan model (tipe native Python — kompatibel PyTorch 2.6+)
torch.save({
    "model_state_dict": cdcn.state_dict(),
    "theta":        float(THETA),
    "acer":         float(best_acer),
    "optimal_thr":  float(OPTIMAL_THR),
    "cdcn_metrics": {k:float(v) for k,v in mc_opt.items()},
    "base_metrics": {k:float(v) for k,v in mb.items()},
}, OUT+"/cdcn_final.pth")

def clean_hist(h):
    return [{k:float(v) for k,v in e.items()} for e in h]

with open(OUT+"/history.json","w") as f:
    json.dump({"cdcn":{"train":clean_hist(hist_c["train"]),"val":clean_hist(hist_c["val"])},
               "baseline":{"train":clean_hist(hist_b["train"]),"val":clean_hist(hist_b["val"])}},
              f, indent=2)

for name in ["eda","augmentasi","training_curves","eval_results",
             "overfit_eval","ablation","inference_upload"]:
    src = f"/content/{name}.png"
    if os.path.exists(src):
        shutil.copy(src, f"{OUT}/{name}.png")

subprocess.run(["zip","-r","/content/cdcn_results.zip","output_cdcn/"],
               cwd="/content",capture_output=True)

print("📦 Output:")
for f_ in sorted(os.listdir(OUT)):
    print(f"  {f_:<30} ({os.path.getsize(f'{OUT}/{f_}')//1024:.0f} KB)")

print("\n📥 Download...")
files.download("/content/cdcn_results.zip")

---
## 🎓 Ringkasan Pipeline

| Bagian | Deskripsi | Yang Dipelajari |
|--------|-----------|-----------------|
| 0 | Setup | Environment GPU |
| 1 | Dataset | Cara akses Kaggle |
| 2 | EDA + Video-Level Split | Memahami data, mencegah data leakage |
| 3 | Preprocessing | Resize, normalisasi, augmentasi |
| 4 | Arsitektur CDCN | CDC layer, residual block, dual output |
| 5 | Training | Loss function, early stopping, metrik ACER |
| 6 | Visualisasi Kurva | Membaca kurva training |
| 7 | Evaluasi Test Set | ROC, Confusion Matrix, kalibrasi threshold |
| 8 | Overfitting Analysis | Bandingkan train vs test |
| 9 | Ablation Study | Validasi nilai θ optimal |
| 10 | Inference | Live demo webcam + upload |
| 11 | Simpan | Download semua output |

## 📚 Referensi

1. Yu, Z. et al. (2020). *Searching Central Difference Convolutional Networks for Face Anti-Spoofing.* CVPR 2020. https://arxiv.org/pdf/2003.04092
2. Boulkenafet, Z. et al. (2017). *OULU-NPU: A Mobile Face Presentation Attack Database.* IEEE FG 2017.
3. Trainingdata Pro (2023). *Real vs Fake Anti-Spoofing Dataset.* Kaggle.
